In [1]:
import warnings
from tqdm import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
from ms_var_prediction import (
    OUTPUT_DIR,
    GaussianMixtureVaR,
    MarkovSwitchingVaR,
    batch_backtest_model,
    fetch_valid_sp500_tickers,
    Settings,
)
from typing import List
from yfinance import Ticker

In this notebook we will run a Value-at-Risk backtest for `GaussianMixtureModel` from `sklearn` and our implentation of `MarkovSwitchingModel` and compare the results.

# Implementation

In [7]:
def run_gmm(tickers: List[Ticker], settings: Settings) -> None:
    gmm_model = GaussianMixtureVaR(max_iter=1000)
    gmm_res = batch_backtest_model(
        gmm_model,
        tickers,
        settings.START_DATE,
        settings.END_DATE,
        settings.ALPHA,
        settings.WINDOW_SHAPE,
    )
    gmm_res.to_csv(
        OUTPUT_DIR / f"backtesting_gmm_p{int(100 * settings.ALPHA)}_{settings.START_DATE}_{settings.END_DATE}"
    )
    return gmm_res

def run_msm(tickers: List[Ticker], settings: Settings) -> None:
    msm_model = MarkovSwitchingVaR(optimizer_options={"max_iter": 1000})
    msm_res = batch_backtest_model(
        msm_model,
        tickers,
        settings.START_DATE,
        settings.END_DATE,
        settings.ALPHA,
        settings.WINDOW_SHAPE,
    )
    msm_res.to_csv(
        OUTPUT_DIR / f"backtesting_msm_p{int(100 * settings.ALPHA)}_{settings.START_DATE}_{settings.END_DATE}"
    )
    return msm_res

def run_backtest(settings: Settings):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    tickers = fetch_valid_sp500_tickers(settings.START_DATE, settings.END_DATE)
    msm_res_df = run_msm(tickers, settings)
    gmm_res_df = run_gmm(tickers, settings)

    return gmm_res_df, msm_res_df

# Backtest run

Let's download the data for the 2010-2025 years and run our models on this period. We will use only the tickers which have the full history on the selected time range. Our backtest will be separate for `alpha=0.01` and `alpha=0.05`.

In [8]:
start_date = "2010-01-01"
end_date = "2026-01-01"
alpha = 0.05
window_shape = 250

settings = Settings(
    START_DATE=start_date,
    END_DATE=end_date,
    ALPHA=alpha,
    WINDOW_SHAPE=window_shape
)

gmm_res_p5_df, msm_res_p5_df = run_backtest(settings)

print(f"Results for GMM: {gmm_res_p5_df.sum().to_dict()}")
print(f"Results for MSM: {msm_res_p5_df.sum().to_dict()}")

Yahoo Finance Tickers Backtested:   0%|          | 0/498 [00:00<?, ?it/s]

Yahoo Finance Tickers Backtested: 100%|██████████| 498/498 [5:30:19<00:00, 39.80s/it]  

Results for GMM: {'binomial': 488, 'independence_simple': 66, 'kupiec': 488, 'christoffersen_independence': 64, 'christoffersen_conditional': 85}
Results for MSM: {'binomial': 441, 'independence_simple': 379, 'kupiec': 440, 'christoffersen_independence': 376, 'christoffersen_conditional': 366}


In [9]:
start_date = "2010-01-01"
end_date = "2026-01-01"
alpha = 0.01
window_shape = 250

settings = Settings(
    START_DATE=start_date,
    END_DATE=end_date,
    ALPHA=alpha,
    WINDOW_SHAPE=window_shape
)

gmm_res_p1_df, msm_res_p1_df = run_backtest(settings)

print(f"Results for GMM: {gmm_res_p1_df.sum().to_dict()}")
print(f"Results for MSM: {msm_res_p1_df.sum().to_dict()}")

Yahoo Finance Tickers Backtested: 100%|██████████| 498/498 [6:26:14<00:00, 46.54s/it]  

Results for GMM: {'binomial': 303, 'independence_simple': 233, 'kupiec': 337, 'christoffersen_independence': 233, 'christoffersen_conditional': 204}
Results for MSM: {'binomial': 234, 'independence_simple': 266, 'kupiec': 266, 'christoffersen_independence': 267, 'christoffersen_conditional': 194}


Now we can check the result.

**Conclusion.** We can see that `MarkovSwitchingModel` achieves a significant improvement on independence tests for `alpha=0.05` due to its ability of catching the regime, loosing a bit on binomial and kupiec tests. For `alpha=0.01` we see a similar trend, however both models perform quite poorly and the difference is not as outstanding.